# AI Circular Economy Demand Forecasting — Time-Series Pipeline

**Goal**: Forecast upcoming regional and material demand volumes (kg) using lag features and rolling window aggregates.


## 1. Data Ingestion & Lag Engineering


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

df = pd.read_csv('../datasets/material_demand.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['material_type', 'date']).reset_index(drop=True)

# Engineering Lags & Rolling Window
groups = df.groupby('material_type')['quantity_requested']
df['lag_1'] = groups.shift(1)
df['lag_7'] = groups.shift(7)
df['lag_30'] = groups.shift(30)
df['rolling_mean_7'] = groups.transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_mean_30'] = groups.transform(lambda x: x.shift(1).rolling(30).mean())
df_clean = df.dropna().reset_index(drop=True)
df_clean.head()


## 2. Model Training & Forecast Evaluation


In [ ]:
split_idx = int(len(df_clean) * 0.8)
train = df_clean.iloc[:split_idx]
test = df_clean.iloc[split_idx:]

features = ['lag_1', 'lag_7', 'lag_30', 'rolling_mean_7', 'rolling_mean_30']
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(train[features], train['quantity_requested'])

preds = model.predict(test[features])
print(f'Demand Forecast Test MAE: {mean_absolute_error(test["quantity_requested"], preds):.2f} kg')
print(f'Demand Forecast Test R²:  {r2_score(test["quantity_requested"], preds):.4f}')
